In [2]:
import os, sys, cdsapi, calendar, shutil
sys.path.append('backend/app/')
from backend.app.services import functions
from datetime import datetime
import xarray as xr, numpy as np, pandas as pd
from dateutil.relativedelta import relativedelta

In [5]:
path = r"backend\src\samples\grid\Grid_net.nc"
with xr.open_dataset(path) as ds:
    gdf = functions.unstructuredGridCreator(ds)

In [7]:
gdf.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

# Download meteo data

In [21]:
# Setup variables
variables = {
    '2m_temperature': 't2m', # Air Temperature
    '2m_dewpoint_temperature': 'd2m', # Dew Point Temperature
    'total_cloud_cover': 'tcc', # Cloud Cover
    'surface_solar_radiation_downwards': 'ssrd', # Shortwave radiation
    '10m_u_component_of_wind': 'u10', '10m_v_component_of_wind': 'v10', # Wind
}
start, end = '2023-01-01 00:00:00', '2023-01-02 00:00:00'
start_time = datetime.strptime(start, '%Y-%m-%d %H:%M:%S')
end_time = datetime.strptime(end, '%Y-%m-%d %H:%M:%S')
dataset, resolution, delta = 'reanalysis-era5-single-levels', 0.25, 0.125
lat, lon = 62.458590134021, 6.355150938034
lat_new = round(lat / resolution) * resolution
lon_new = round(lon / resolution) * resolution
area = [lat_new + delta, lon_new - delta, lat_new - delta, lon_new + delta]
download_dir = os.path.join('test', 'download')
os.makedirs(download_dir, exist_ok=True)
weather = pd.DataFrame()

In [22]:
client = cdsapi.Client(quiet=False, debug=False)
current = start_time.replace(day=1)
while current <= end_time:
    year, month = current.year, current.month
    last_day = calendar.monthrange(year, month)[1]
    month_start = datetime(year, month, 1)
    month_end = datetime(year, month, last_day, 23)
    # Clip by requested range
    actual_start = max(start_time, month_start)
    actual_end = min(end_time, month_end)
    # Days to download
    days = [f"{d:02d}" for d in range(actual_start.day, actual_end.day + 1)]
    df_temp = pd.DataFrame()
    for key, var in variables.items():
        request = {
            'product_type': 'reanalysis', 'variable': [key],
            'year': [str(year)], 'month': [f"{month:02d}"], 'day': days,
            'time': [f"{h:02d}:00" for h in range(24)], 'area': area,
            'data_format': 'netcdf', 'download_format': 'unarchived'
        }
        out_file = f"{year}_{month:02d}_{var}.nc"
        out_path = os.path.join(download_dir, out_file)
        client.retrieve(dataset, request, out_path)
        with xr.open_dataset(out_path) as ds:
            df = pd.DataFrame(index=pd.to_datetime(ds['valid_time'].values))
            df[var] = ds[var].values.flatten()
        df_temp = pd.concat([df_temp, df], axis=1)
        functions.safe_remove(out_path)
    weather = pd.concat([weather, df_temp], axis=0)
    current += relativedelta(months=1)
shutil.rmtree(download_dir)

2026-07-06 21:01:57,722 INFO Request ID is 4ed2b961-3010-41f9-9be5-b3c4ab40bb69
2026-07-06 21:01:57,821 INFO status has been updated to accepted
2026-07-06 21:02:06,337 INFO status has been updated to running
2026-07-06 21:02:13,306 INFO status has been updated to successful


5192946127401c6a1273713e759cd231.nc:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

2026-07-06 21:02:15,524 INFO Request ID is f4030e1d-8ea2-47cf-8fbb-2fe0a793ac94
2026-07-06 21:02:15,602 INFO status has been updated to accepted
2026-07-06 21:02:29,254 INFO status has been updated to running
2026-07-06 21:02:36,941 INFO status has been updated to successful


2b2c61833d7960c123865e2823b2e9c.nc:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

2026-07-06 21:02:39,592 INFO Request ID is 039c7df2-1aad-4caf-8d60-97b9628e407a
2026-07-06 21:02:39,706 INFO status has been updated to accepted
2026-07-06 21:03:01,254 INFO status has been updated to successful


...

f77e811e3a24c6733d93062160af5536.nc:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

2026-07-06 21:03:32,111 INFO Request ID is 18a4a537-7000-4460-a0e8-2e61cb2da99b
2026-07-06 21:03:32,344 INFO status has been updated to accepted
2026-07-06 21:03:49,694 INFO status has been updated to running
2026-07-06 21:03:58,265 INFO status has been updated to successful


d2596d01cf2727d4012212bc5c6f166e.nc:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

2026-07-06 21:04:00,096 INFO Request ID is ab5f7293-9901-4ce7-abed-975e40ea3823
2026-07-06 21:04:00,858 INFO status has been updated to accepted
2026-07-06 21:04:14,525 INFO status has been updated to running
2026-07-06 21:04:22,199 INFO status has been updated to successful


780caf71c4c6466e1f740346749bcb1d.nc:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

In [55]:
df = weather.copy()
df['t2m'], df['d2m'] = df['t2m'] - 273.15, df['d2m'] - 273.15
df['Magnitude [m/s]'] = np.sqrt(df['u10']**2 + df['v10']**2)
angle = (np.degrees(np.arctan2(-df['u10'], -df['v10'])) + 360) % 360
df['Angle [deg]'] = angle.round(1)
df['ssrd'], df['tcc'] = df['ssrd']/3600, df['tcc']*100
es = 6.112 * np.exp((17.67 * df['t2m']) / (df['t2m'] + 243.5))
e = 6.112 * np.exp((17.67 * df['d2m']) / (df['d2m'] + 243.5))
df['Humidity [%]'] = np.clip(100 * e / es, 0, 100)
df = df.drop(columns=['u10', 'v10', 'd2m'], axis=0)
new_columns = {'t2m':'Air temperature [°C]', 'tcc': 'Cloud coverage [%]', 'ssrd': 'Solar radiation [W/m2]'}
df = df.rename(columns=new_columns)
df = df[['Humidity [%]', 'Air temperature [°C]', 'Cloud coverage [%]', 
    'Solar radiation [W/m2]', 'Magnitude [m/s]', 'Angle [deg]']]
df.index.name = 'Time'

In [56]:
df

,Humidity [%],Air temperature [°C],Cloud coverage [%],Solar radiation [W/m2],Magnitude [m/s],Angle [deg]
Time,,,,,,
2023-01-01 00:00:00,73.113739,0.797516,60.034180,0.000000,5.918163,272.399994
2023-01-01 01:00:00,69.380974,0.622711,54.730225,0.000000,4.864110,257.500000
2023-01-01 02:00:00,64.222183,0.425201,46.353149,0.000000,4.240874,243.399994
2023-01-01 03:00:00,61.721813,0.463287,35.476685,0.000000,3.887233,227.000000
2023-01-01 04:00:00,66.044426,0.470612,52.890015,0.000000,3.911917,208.399994
2023-01-01 05:00:00,74.161194,0.042633,28.900146,0.000000,4.245868,188.100006
2023-01-01 06:00:00,79.287949,-0.379974,27.447510,0.000000,4.547823,177.899994
2023-01-01 07:00:00,79.418480,-0.818207,5.435181,0.000000,4.786444,172.899994
2023-01-01 08:00:00,78.304634,-1.081146,4.754639,0.000000,4.843805,171.199997


2023-01-01 00:00:00    5.918163
2023-01-01 01:00:00    4.864110
2023-01-01 02:00:00    4.240874
2023-01-01 03:00:00    3.887233
2023-01-01 04:00:00    3.911917
2023-01-01 05:00:00    4.245868
2023-01-01 06:00:00    4.547823
2023-01-01 07:00:00    4.786444
2023-01-01 08:00:00    4.843805
2023-01-01 09:00:00    4.918464
2023-01-01 10:00:00    4.265143
2023-01-01 11:00:00    3.956651
2023-01-01 12:00:00    4.112846
2023-01-01 13:00:00    4.167901
2023-01-01 14:00:00    4.228968
2023-01-01 15:00:00    4.491441
2023-01-01 16:00:00    4.959171
2023-01-01 17:00:00    5.098366
2023-01-01 18:00:00    5.099139
2023-01-01 19:00:00    5.256403
2023-01-01 20:00:00    5.192173
2023-01-01 21:00:00    4.998812
2023-01-01 22:00:00    4.737221
2023-01-01 23:00:00    4.628104
2023-01-02 00:00:00    4.608849
2023-01-02 01:00:00    4.591264
2023-01-02 02:00:00    4.638174
2023-01-02 03:00:00    4.694841
2023-01-02 04:00:00    4.791249
2023-01-02 05:00:00    5.009550
2023-01-02 06:00:00    5.081351
2023-01-